# 05 — Score-valley support and resistance bands

Same saved 2,000 trades and step-one differences. Exact-price scores define peaks and neighboring valleys. No adaptive gap, movement half-life, smoothing or width multiplier. Intermediate tables expose every boundary decision.


In [ ]:
# Basic setup. Generated data and charts must stay outside the repository.
import os
import sys
os.environ["PYTHONDONTWRITEBYTECODE"] = "1"
sys.dont_write_bytecode = True
from pathlib import Path
from datetime import datetime, timezone
import json
import hashlib
import subprocess
import numpy as np
import polars as pl
import matplotlib.pyplot as plt
from IPython.display import display

REPO = Path("D:/TradingCodes/quant-research-workbench")
RUNTIME_ROOT = Path("D:/TradingML/runtimes")
SOURCE_RUN = RUNTIME_ROOT / "level-book-v8/reaction-demo/20261020T204419563339Z"
EXPECTED_SOURCE_HASH = "b40270ba8fc4055d515fa9061ce5333cce152890de23e249dbbc827e7e0ec70d"
N_EVENTS = 2000
# Explicit illustrative settings, fixed before inspecting the result.
TAU_SECONDS = 1.0          # Same 1-second reference used in our discussion; not a candle timeframe.
SCORE_THRESHOLD = 0.60     # A display/selection rule, not a calibrated confidence threshold.
WEIGHT_COUNT = 1.0 / 3     # Equal weights reproduce (normalized count + volume + time) / 3.
WEIGHT_VOLUME = 1.0 / 3
WEIGHT_TIME = 1.0 / 3
assert TAU_SECONDS > 0
assert 0 < SCORE_THRESHOLD <= 1
assert min(WEIGHT_COUNT, WEIGHT_VOLUME, WEIGHT_TIME) >= 0
assert np.isclose(WEIGHT_COUNT + WEIGHT_VOLUME + WEIGHT_TIME, 1.0)
assert REPO.is_dir() and RUNTIME_ROOT.is_dir(), "Required root unavailable"
RUN = RUNTIME_ROOT / "level-book-v8/score-valley-bands" / datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
RUN.mkdir(parents=True, exist_ok=False)
pl.Config.set_tbl_rows(20)
pl.Config.set_tbl_cols(16)
print("Output directory:", RUN)


## 1. Load and verify the original sample
This reads the saved canonical-derived snapshot, not raw SIP flatfiles. We validate its original content hash before taking the requested prefix. If it is missing or changed, stop rather than silently fetching a different sample.

The original extraction excluded delayed/unknown-clock and condition-ineligible trades. Its audit and provenance are retained. Price is stored in exact units of 1/10,000 dollar. Ordinal is metadata, not a predictor: order remains `(SIP timestamp, ordinal)`, exactly as in notebook 01. Time differences can be zero for equal timestamps.

We slice **before** calculating shifts. The first event outside the prefix must not reveal the final event's departure.

In [ ]:
# Recreate the original loader's hashed records, in their original order.
source = pl.read_parquet(SOURCE_RUN / "source_trades.parquet")
provenance = json.loads((SOURCE_RUN / "source_provenance.json").read_text(encoding="utf-8"))
source_records = source.select("ordinal", "time_us", "price_units", "volume").to_dicts()
# ClickHouse JSON encoded integral sizes as integers; Parquet promoted the
# mixed size column to Float64. Restore only that JSON representation for hashing.
for record in source_records:
    if float(record["volume"]).is_integer():
        record["volume"] = int(record["volume"])
source_hash = hashlib.sha256(json.dumps(source_records, sort_keys=True).encode()).hexdigest()
assert source_hash == provenance["sample_sha256"] == EXPECTED_SOURCE_HASH
assert source.height == 2000 and provenance["ticker"] == "AAPL"
assert provenance["day"] == "2026-08-20"
assert 3 <= N_EVENTS <= source.height
assert source["ordinal"].n_unique() == source.height
assert source.select("time_us", "ordinal").equals(source.sort("time_us", "ordinal").select("time_us", "ordinal"))
raw = source.head(N_EVENTS)
# These are the three original quantities; event and ordinal identify rows.
display(raw.select("event", "ordinal", "time", "price", "volume").head(15))
print("Selected events:", raw.height, "unique prices:", raw["price_units"].n_unique())
print("Time span:", raw["time"][0], "to", raw["time"][-1])
display(pl.DataFrame(provenance["audit"]))

## 2. Our agreed step one: six vectorized differences
For any column x:

- backward = current minus previous;
- forward = next minus current.

Time differences are seconds, price differences dollars, and volume differences changes in individual trade size. Arrival and departure are the signs of the respective price differences. A zero price difference means unchanged, not missing.

The first backward values and final forward values remain null. Forward time is the current trade's completed hold contribution. We do not add elapsed time for the pending final row or pretend its reaction is unchanged.

In [ ]:
def step_one(frame):
    """Derive adjacent-row differences on precisely the supplied prefix."""
    return frame.with_columns(
        ((pl.col("time_us") - pl.col("time_us").shift(1)) / 1e6).alias("time_backward"),
        ((pl.col("time_us").shift(-1) - pl.col("time_us")) / 1e6).alias("time_forward"),
        ((pl.col("price_units") - pl.col("price_units").shift(1)) / 10000).alias("price_backward"),
        ((pl.col("price_units").shift(-1) - pl.col("price_units")) / 10000).alias("price_forward"),
        (pl.col("volume") - pl.col("volume").shift(1)).alias("volume_backward"),
        (pl.col("volume").shift(-1) - pl.col("volume")).alias("volume_forward"),
    ).with_columns(
        pl.col("price_backward").sign().cast(pl.Int8).alias("arrival"),
        pl.col("price_forward").sign().cast(pl.Int8).alias("departure"),
    )

step1 = step_one(raw)
step_columns = ["event", "time", "price", "volume", "time_backward", "time_forward",
                "price_backward", "price_forward", "volume_backward", "volume_forward"]
display(step1.select(step_columns).head(15))
display(step1.select(step_columns).tail(3))

## 3. Group by exact price and the nine direction pairs

| Arrival / departure | Down (−1) | Unchanged (0) | Up (+1) |
|---|---|---|---|
| Down (−1) | Down continuation | Pause after decline | Support-like bounce |
| Unchanged (0) | Down after unchanged | Unchanged / unchanged | Up after unchanged |
| Up (+1) | Resistance-like rejection | Pause after rise | Up continuation |

Each trade belongs to one cell when both directions are known. There are no visit IDs. For example, down→unchanged followed by unchanged→up is **not** merged into one bounce.

Each cell stores count, current traded volume, and sums of all six differences. Dividing a difference sum by count gives its cell mean. Signed volume changes can cancel, so they remain explanatory measurements, not an automatic strength score.

Boundary rows are displayed separately and still contribute to the total activity distribution.

In [ ]:
# These are the six differences already calculated in step one.
# Listing their names lets us apply the same sum operation to each column.
difference_columns = ["time_backward", "time_forward", "price_backward", "price_forward", "volume_backward", "volume_forward"]
# Keep rows with BOTH directions known so each can be assigned to one of nine groups.
# pl.col selects a column, is_not_null checks availability, and & requires both checks.
# A direction of zero (unchanged) is valid and is retained.
complete = step1.filter(pl.col("arrival").is_not_null() & pl.col("departure").is_not_null())
# Retain rows missing either direction in a separate audit table; | means OR.
# The first arrival and final departure are unknown, not unchanged.
# Keeping them separately prevents silent loss of their count and volume.
boundary_rows = step1.filter(pl.col("arrival").is_null() | pl.col("departure").is_null())
# Group by exact integer price AND arrival AND departure.
# For example, all rows at one price with arrival=-1 and departure=+1 form one group.
# Integer price units avoid floating-point equality problems. agg calculates one row per group.
profile = complete.group_by("price_units", "arrival", "departure").agg(
    # Count the trades in this group. alias gives the result its readable column name.
    # This is a trade count, not a count of independent visits.
    pl.len().alias("trade_count"),
    # Sum CURRENT trade sizes in this group, not next-trade size or volume change.
    # Thus bounce-group volume means volume on trades immediately followed by an upward move.
    pl.col("volume").sum().alias("volume_sum"),
    # Build one sum expression per difference column; * expands the list into agg arguments.
    # Forward-time sum is the group's completed hold contribution, measured in seconds.
    # Backward time is kept separately: adding the two time sums would double-count intervals.
    # Price and volume differences are signed; positive and negative values can cancel.
    # Dividing each sum by trade_count yields that group's mean difference.
    *[pl.col(name).sum().alias(name + "_sum") for name in difference_columns],
# Close the aggregation, add a dollar-price column for display, then sort the rows.
# Grouping remains on price_units; dividing by 10,000 is presentation only.
# Sorting by price and both directions makes the table easier to inspect.
).with_columns((pl.col("price_units") / 10000).alias("price")).sort("price_units", "arrival", "departure")
# Display the first 18 aggregated rows only; the full profile remains in memory.
# This limits screen output, not the data used by subsequent calculations.
display(profile.head(18))
# Report how many original trade rows were classified and how many have unknown boundaries.
# DataFrame.height is its number of rows; these two counts should total N_EVENTS.
print("Classified rows:", complete.height, "boundary rows:", boundary_rows.height)
# Show the original event identities and known directions of the boundary rows.
# select chooses display columns; it does not modify or discard the saved boundary table.
display(boundary_rows.select("event", "price", "volume", "arrival", "departure"))

# Separately aggregate ALL selected trades by price, including the boundary rows.
# Activity totals need no next-trade outcome, so their count and volume are already known.
# This table describes overall trading concentration, not a particular reaction.
activity = raw.group_by("price_units").agg(pl.len().alias("total_trades"), pl.col("volume").sum().alias("total_volume")).sort("price_units")
# Rank prices by total trade count descending; break ties by lower price first.
# Select the first price_units value to choose one reproducible example for the matrix.
# "Busiest" here means most trades, not greatest volume or strongest support.
busiest = activity.sort(["total_trades", "price_units"], descending=[True, False])["price_units"][0]
# Explicitly create all nine (arrival, departure) pairs: each direction is -1, 0 or +1.
# The departure list repeats three times, once for each arrival direction.
# Int8 matches the direction-column types in profile so the join uses compatible keys.
grid = pl.DataFrame({"arrival": [-1,-1,-1,0,0,0,1,1,1], "departure": [-1,0,1]*3}).cast(pl.Int8)
# Filter the profile to the chosen price, then match rows by both directions.
# A LEFT join preserves all nine grid rows, even if a combination never occurred.
# An unobserved combination has null aggregate values after this join.
matrix = grid.join(profile.filter(pl.col("price_units") == busiest), on=["arrival", "departure"], how="left")
# Print the chosen price in dollars so the displayed matrix is clearly identified.
print("Nine cells at price:", busiest / 10000)
# Display directions plus count, volume and completed hold time for each of nine cells.
# Here null means NO MATCHING GROUP, so those three totals are correctly displayed as zero.
# This does not replace unknown directions or pending forward differences in the source.
# fill_null affects this display expression; the original matrix/profile remain unchanged.
display(matrix.select("arrival", "departure", pl.col("trade_count", "volume_sum", "time_forward_sum").fill_null(0)))


## 4. Score each exact price first
Use the same equal-weight count, volume and time formula. Count and volume are normalized separately across prices for each role. Quick reversals get `tau / (tau + mean_forward_delay)`. Unknown boundaries are excluded from reaction scores; zero evidence gives score zero. Scores are relative strengths, not probabilities.

Only support (-1,+1) and resistance (+1,-1) define bands here. Unchanged and continuation observations remain in the original nine-cell table. The time reference tau remains 1 second; it is not a band-width parameter.


In [ ]:
# Define the four roles and their time interpretation explicitly.
roles = pl.DataFrame({
    "role": ["support", "resistance"],
    "arrival": [-1, 1],
    "departure": [1, -1],
    "time_preference": ["quick", "quick"],
}).with_columns(pl.col("arrival", "departure").cast(pl.Int8))

def score_bands(price_activity, reaction_profile, tau, weights):
    """Score every band/role, including empty cells. No neighbor or peak tests."""
    assert np.isfinite(tau) and tau > 0
    assert len(weights) == 3 and min(weights) >= 0 and np.isclose(sum(weights), 1)
    # Cross join creates both roles at every band, even when a cell is absent.
    table = price_activity.join(roles, how="cross").join(
        reaction_profile.select("band_id", "arrival", "departure", "trade_count", "volume_sum", "time_forward_sum"),
        on=["band_id", "arrival", "departure"], how="left",
    ).with_columns(
        # Missing group means zero observations/totals, not an unknown source direction.
        pl.col("trade_count", "volume_sum", "time_forward_sum").fill_null(0),
    ).with_columns(
        # over('role') computes each role's maximum across bands, without merging roles.
        pl.col("trade_count").max().over("role").alias("max_count"),
        pl.col("volume_sum").max().over("role").alias("max_volume"),
        # No observations means no known mean delay; leave it null rather than set it to zero.
        pl.when(pl.col("trade_count") > 0).then(pl.col("time_forward_sum") / pl.col("trade_count"))
          .otherwise(None).alias("mean_delay_seconds"),
    ).with_columns(
        # If a role has no evidence anywhere, its count/volume components are zero.
        pl.when(pl.col("max_count") > 0).then(pl.col("trade_count") / pl.col("max_count"))
          .otherwise(0.0).alias("count_component"),
        pl.when(pl.col("max_volume") > 0).then(pl.col("volume_sum") / pl.col("max_volume"))
          .otherwise(0.0).alias("volume_component"),
        # Select QUICK for reversal roles and LONG for continuation roles.
        pl.when(pl.col("trade_count") == 0).then(0.0)
          .when(pl.col("time_preference") == "quick").then(tau / (tau + pl.col("mean_delay_seconds")))
          .otherwise(pl.col("mean_delay_seconds") / (tau + pl.col("mean_delay_seconds"))).alias("time_component"),
    ).with_columns(
        # Expose the contribution of each term so the final score is easy to audit.
        (weights[0] * pl.col("count_component")).alias("count_contribution"),
        (weights[1] * pl.col("volume_component")).alias("volume_contribution"),
        (weights[2] * pl.col("time_component")).alias("time_contribution"),
    ).with_columns(
        # An unobserved cell cannot earn a score from a time bonus.
        pl.when(pl.col("trade_count") > 0)
          .then(pl.col("count_contribution") + pl.col("volume_contribution") + pl.col("time_contribution"))
          .otherwise(0.0).alias("score"),
    )
    return table.sort(["role", "score", "band_id"], descending=[False, True, False])


In [ ]:
# Reuse the scorer with an exact price as the grouping key.
price_scores = score_bands(activity.rename({"price_units":"band_id"}),
    profile.rename({"price_units":"band_id"}), TAU_SECONDS,
    (WEIGHT_COUNT, WEIGHT_VOLUME, WEIGHT_TIME)).rename({"band_id":"price_units"})
price_scores = price_scores.with_columns((pl.col("price_units")/10000).alias("price"))
display(price_scores.select("role","price","trade_count","volume_sum","mean_delay_seconds",
    "count_component","volume_component","time_component","score").sort("role","price").head(30))


## 5. Locate peaks, plateaus and valleys
Sort all observed prices separately for each role, including zero-score prices. Consecutive exactly equal scores form one plateau. Compare each plateau with the previous and next plateau: higher than both is a peak; lower than both is a valley. Plateau representatives use the lower middle observed price when there are two middle choices.

An endpoint higher than its sole neighbor is an **edge peak**. Its outer boundary is open at the observed price-range edge. A wholly flat profile has no distinguishable peak and creates no bands. No rounding tolerance or smoothing is introduced.

Between consecutive peaks, choose the lowest-score plateau (the lower-priced one on an exact tie). Its representative price is the boundary. A boundary is drawn on both neighboring bands, but its evidence belongs only to the band on its right. The outer bands extend to the observed range edges. These are boundaries in the observed profile, not proof that all intermediate prices traded. Large unobserved gaps remain visible through member counts and width.


In [ ]:
def find_basins(prices, scores):
    """Deterministic one-dimensional partition; returns audit tables as records."""
    prices = list(map(int, prices)); scores = list(map(float, scores))
    assert len(prices) == len(scores) and len(prices) > 0
    assert all(a < b for a,b in zip(prices,prices[1:]))
    plateaus = []
    # A run of equal neighboring scores gets one representative, avoiding duplicate peaks.
    for i, value in enumerate(scores):
        if not plateaus or value != plateaus[-1]["score"]:
            plateaus.append(dict(start=i,end=i,score=value))
        else:
            plateaus[-1]["end"] = i
    for j,p in enumerate(plateaus):
        left = plateaus[j-1]["score"] if j else None
        right = plateaus[j+1]["score"] if j+1<len(plateaus) else None
        neighbors = [v for v in (left,right) if v is not None]
        p.update(plateau_id=j, representative=(p["start"]+p["end"])//2,
                 previous_score=left,next_score=right,
                 kind="peak" if neighbors and all(p["score"]>v for v in neighbors)
                      else "valley" if left is not None and right is not None and p["score"]<min(left,right)
                      else "flat" if not neighbors else "slope")
        p["price"] = prices[p["representative"]]/10000
    peaks = [p for p in plateaus if p["kind"]=="peak"]
    cuts = []
    for a,b in zip(peaks,peaks[1:]):
        between = plateaus[a["plateau_id"]+1:b["plateau_id"]]
        valley = min(between,key=lambda p:(p["score"],p["representative"]))
        cuts.append(valley["representative"])
    bands=[]; membership=[]
    for j,peak in enumerate(peaks):
        left = cuts[j-1] if j else 0
        right = cuts[j] if j<len(cuts) else len(prices)-1
        # A shared valley belongs to the right band; the last band includes the last price.
        stop = right if j<len(cuts) else len(prices)
        bands.append(dict(band_id=j,lower_units=prices[left],upper_units=prices[right],
            peak_units=prices[peak["representative"]],peak_score=peak["score"],
            lower_reason="valley" if j else "open range edge",
            upper_reason="valley" if j<len(cuts) else "open range edge",
            member_prices=stop-left))
        membership.extend(dict(price_units=prices[i],band_id=j) for i in range(left,stop))
    return plateaus,bands,membership

# Hand example from our discussion: the 100.04 valley divides the two peaks.
example_prices=[1000000+i*100 for i in range(8)]
example_scores=[.10,.35,.85,.60,.15,.50,.80,.20]
example_plateaus,example_bands,example_members=find_basins(example_prices,example_scores)
display(pl.DataFrame({"price":np.array(example_prices)/10000,"score":example_scores}))
display(pl.DataFrame(example_plateaus))
display(pl.DataFrame(example_bands))
display(pl.DataFrame(example_members).with_columns((pl.col("price_units")/10000).alias("price")))


## 6. Apply the boundaries and recompute band strength
The price score determines the shape of each role's bands. After partitioning, sum original count, current volume and forward-time evidence within each band, then recompute the same normalized score across bands of that role. Do not average price scores. The score at the peak is retained separately for comparison.

Support and resistance have independent partitions. Their evidence is conserved independently, not added across roles. The complete nine-cell price profile remains available; each band's score uses its own reversal cell only. The band start event is the earliest occurrence of an assigned member price.


In [ ]:
audit_rows=[]; definitions=[]; assignments=[]
for role in ("support","resistance"):
    ordered=price_scores.filter(pl.col("role")==role).sort("price_units")
    plateaus,bands,members=find_basins(ordered["price_units"],ordered["score"])
    audit_rows.extend(dict(role=role,**r) for r in plateaus)
    definitions.extend(dict(role=role,**r) for r in bands)
    assignments.extend(dict(role=role,**r) for r in members)
plateau_audit=pl.DataFrame(audit_rows)
boundaries=pl.DataFrame(definitions, schema_overrides={"band_id":pl.Int64})
membership=pl.DataFrame(assignments, schema_overrides={"price_units":pl.Int64,"band_id":pl.Int64})
# This sample has non-flat profiles; the flat-profile behavior is demonstrated and tested below.
assert boundaries.height > 0
contributions=membership.join(price_scores,on=["role","price_units"],how="left")
first_seen=raw.group_by("price_units").agg(pl.col("event").min().alias("first_event"))
band_scores=contributions.join(first_seen,on="price_units").group_by("role","band_id").agg(
    pl.col("trade_count").sum(),pl.col("volume_sum").sum(),pl.col("time_forward_sum").sum(),
    pl.col("first_event").min(),
).with_columns(
    (pl.col("trade_count")/pl.col("trade_count").max().over("role")).alias("count_component"),
    (pl.col("volume_sum")/pl.col("volume_sum").max().over("role")).alias("volume_component"),
    (pl.col("time_forward_sum")/pl.col("trade_count")).alias("mean_delay_seconds"),
).with_columns((TAU_SECONDS/(TAU_SECONDS+pl.col("mean_delay_seconds"))).alias("time_component"))
band_scores=band_scores.with_columns(
    ((pl.col("count_component")+pl.col("volume_component")+pl.col("time_component"))/3).alias("score")
).join(boundaries,on=["role","band_id"]).sort("role","lower_units")
# Full audit data is saved; displayed rows focus on the first few boundary decisions.
print("Plateau comparisons (full table: plateau_audit):")
display(plateau_audit.head(30))
print("Peaks and valleys:")
display(plateau_audit.filter(pl.col("kind").is_in(["peak","valley"])).head(30))
print("Boundary reasons and original member-price counts:")
display(boundaries.head(20))
print("Exact-price evidence assigned to bands:")
display(contributions.select("role","band_id","price","score","trade_count","volume_sum","time_forward_sum").head(30))
print("Recomputed band strengths:")
display(band_scores.head(20))
for name,table in dict(plateau_audit=plateau_audit,boundaries=boundaries,membership=membership,
                       contributions=contributions,band_scores=band_scores,price_scores=price_scores).items():
    table.write_parquet(RUN/(name+".parquet"))

# Follow one real interior support band from its boundary valleys to its source evidence.
example_band = boundaries.filter((pl.col("role")=="support") &
    (pl.col("lower_reason")=="valley") & (pl.col("upper_reason")=="valley")).head(1)
if example_band.height:
    chosen = example_band.row(0,named=True)
    print("Worked real band: support B", chosen["band_id"], "(upper boundary evidence belongs to the next band)")
    display(example_band)
    display(plateau_audit.filter((pl.col("role")=="support") &
        (pl.col("price")>=chosen["lower_units"]/10000) & (pl.col("price")<=chosen["upper_units"]/10000)))
    display(contributions.filter((pl.col("role")=="support") & (pl.col("band_id")==chosen["band_id"])))
    display(band_scores.filter((pl.col("role")=="support") & (pl.col("band_id")==chosen["band_id"])))


## 7. Inspect both roles together
The top panel overlays support and resistance on the trade-price trace. The lower two panels expose each role's original score-versus-price profile, peak locations and valley boundaries. This makes a chart band traceable to the profile that created it.

Threshold sliders filter the **recomputed band score**; they do not move boundaries or merge bands. Defaults are 0.50 for both roles. Labels contain role, band ID and score. Band boundaries use all selected events; their first-seen starts are retrospective overlays, not historical discovery times. Edge bands have open outer boundaries. Small noisy peaks remain visible rather than being hidden by an extra parameter.


In [ ]:
import ipywidgets as widgets
colors={"support":"#16803c","resistance":"#c53030"}
chart_state={}
def draw_bands(support,resistance):
    thresholds={"support":support,"resistance":resistance}
    selected=band_scores.join(pl.DataFrame({"role":list(thresholds),"threshold":list(thresholds.values())}),on="role").filter(pl.col("score")>=pl.col("threshold"))
    fig=plt.figure(figsize=(18,max(12,.24*selected.height+4)))
    grid=fig.add_gridspec(3,1,height_ratios=[4,1,1],hspace=.38)
    ax=fig.add_subplot(grid[0]); fig.subplots_adjust(right=.78)
    ax.plot(raw["event"],raw["price"],color="black",linewidth=.8,label="Trade price")
    lo,hi=ax.get_ylim(); ordered=selected.with_columns(((pl.col("lower_units")+pl.col("upper_units"))/2).alias("midpoint")).sort("midpoint","role")
    ys=np.linspace(lo+.03*(hi-lo),hi-.03*(hi-lo),ordered.height) if ordered.height>1 else [(lo+hi)/2]*ordered.height
    for row,y in zip(ordered.iter_rows(named=True),ys):
        lower,upper=row["lower_units"]/10000,row["upper_units"]/10000
        color=colors[row["role"]]; end=int(raw["event"][-1])
        ax.fill_between([row["first_event"],end],lower,upper,color=color,alpha=.13)
        ax.hlines([lower,upper],row["first_event"],end,color=color,linewidth=.8,
                  linestyles="--" if row["role"]=="resistance" else "-")
        ax.annotate(f"{row['role']} B{row['band_id']} | {row['score']:.3f}",xy=(end,(lower+upper)/2),
            xytext=(1.02,y),textcoords=ax.get_yaxis_transform(),color=color,fontsize=9,
            arrowprops=dict(arrowstyle="-",color=color,lw=.5,relpos=(0,.5)),annotation_clip=False)
    for role,color in colors.items():
        ax.plot([],[],color=color,label=f"{role}: {selected.filter(pl.col('role')==role).height} bands >= {thresholds[role]:.2f}")
    ax.legend(); ax.grid(alpha=.15)
    ax.set(title="Score-valley support and resistance bands (retrospective)",xlabel="Event number",ylabel="Trade price ($)")
    for i,(role,color) in enumerate(colors.items(),1):
        panel=fig.add_subplot(grid[i])
        p=price_scores.filter(pl.col("role")==role).sort("price_units")
        panel.plot(p["price"],p["score"],color=color,lw=.8)
        peaks=plateau_audit.filter((pl.col("role")==role)&(pl.col("kind")=="peak"))
        panel.scatter(peaks["price"],peaks["score"],color=color,s=10,label="Peaks")
        cuts=boundaries.filter((pl.col("role")==role)&(pl.col("lower_reason")=="valley"))
        panel.vlines(cuts["lower_units"]/10000,0,1,color="gray",alpha=.35,linewidth=.6,label="Valley boundaries")
        panel.set(title=role+" exact-price profile (all boundaries before threshold)",xlabel="Price ($)",ylabel="Price score",ylim=(-.03,1.03))
        panel.legend(fontsize=8); panel.grid(alpha=.12)
    fig.savefig(RUN/"score_valley_bands.png",dpi=140,bbox_inches="tight")
    plt.show(); plt.close(fig)
    selected.write_parquet(RUN/"selected_bands.parquet")
    (RUN/"thresholds.json").write_text(json.dumps(thresholds),encoding="utf-8")
    chart_state.update(selected=selected,thresholds=thresholds,renders=chart_state.get("renders",0)+1)
controls={role:widgets.FloatSlider(value=.5,min=0,max=1,step=.01,description=role.title(),continuous_update=False,layout=widgets.Layout(width="450px")) for role in colors}
output=widgets.interactive_output(draw_bands,controls)
display(widgets.VBox([widgets.HBox(list(controls.values())),output]))


## 8. Check boundaries and accounting
Hand checks include two peaks, tied plateaus, a flat profile, an endpoint peak and unique membership. Real-data checks verify count, volume and hold-time conservation independently for each role. The resulting strengths are descriptive; no predictive evaluation is claimed.


In [ ]:
assert len(example_bands)==2
assert example_bands[0]["upper_units"]==example_bands[1]["lower_units"]==1000400
assert [r["band_id"] for r in example_members]==[0,0,0,0,1,1,1,1]
assert find_basins([1,2,3],[.5,.5,.5])[1]==[]
p,b,m=find_basins([1,2,3,4,5,6,7],[0,1,1,0,0,1,0])
assert len(b)==2 and b[0]["peak_units"]==2 and b[0]["upper_units"]==4
assert len(m)==7 and len({r["price_units"] for r in m})==7
assert find_basins([1,2,3],[.1,.2,.3])[1][0]["peak_units"]==3
assert membership.unique(["role","price_units"]).height==membership.height
for role in colors:
    before=price_scores.filter(pl.col("role")==role)
    after=band_scores.filter(pl.col("role")==role)
    assert membership.filter(pl.col("role")==role).height==before.height
    for col in ("trade_count","volume_sum","time_forward_sum"):
        assert np.isclose(before[col].sum(),after[col].sum()),(role,col)
assert band_scores["score"].is_finite().all()
assert band_scores["score"].min()>=0 and band_scores["score"].max()<=1
manifest=dict(source_sha256=source_hash,events=N_EVENTS,method="score-valley",tau=TAU_SECONDS,
    bands=band_scores.group_by("role").len().to_dicts(),checks="passed",
    git_commit=subprocess.check_output(["git","rev-parse","HEAD"],cwd=REPO,text=True).strip())
(RUN/"manifest.json").write_text(json.dumps(manifest,indent=2),encoding="utf-8")
(RUN/"source_provenance.json").write_text(json.dumps(provenance,indent=2),encoding="utf-8")
print(json.dumps(manifest,indent=2))
